## A machine learning approach to volatility forecasting


###### In this notebook, I try to replicate the research by Christensen, Siggaard and Veliyev (2022). I use EURUSD exchange rate tick data from November 2025-January 2026, so with intraday trading data (bid and ask).

In [ ]:
import numpy as np
import pandas as pd
from numpy.linalg import lstsq

###### First, we load in the exchange rate data, Compute 5-minute log-returns and daily realized variance.

In [ ]:
def build_series(
    files=[
        "EURUSD_November2025.xlsx",
        "EURUSD_December2025.xlsx",
        "EURUSD_January2026.xlsx",
    ],
    rescale_to_spot=True,
    rescale_factor=1e5
):
    dfs=[]
    for f in files:
        df=pd.read_excel(f, engine="openpyxl", header=None)
        
        df.columns=["symbol", "timestamp", "bid", "ask"]

        df["datetime"]=pd.to_datetime(
            df["timestamp"],
            format="%Y%m%d %H:%M:%S.%f",
            errors="raise"
        )
        
        df=df[["symbol", "datetime", "bid", "ask"]]
        dfs.append(df)

    data=pd.concat(dfs, ignore_index=True)
    data=data.sort_values("datetime").set_index("datetime")

    if rescale_to_spot:
        data["bid"]=data["bid"]/rescale_factor
        data["ask"]=data["ask"]/rescale_factor

    data["mid"]=(data["bid"]+data["ask"])/2.0

    mid_5m=data["mid"].resample("5min").last().dropna()

    logret_5m=np.log(mid_5m / mid_5m.shift(1)).dropna()

    # Calculate daily realized variance
    rv_daily=logret_5m.pow(2).resample("1D").sum().dropna()
    rv_daily=rv_daily.replace(0, np.nan).dropna()
    rv_daily=rv_daily[rv_daily > 0]
    rv_daily=rv_daily.sort_index()

    return data, mid_5m, logret_5m, rv_daily

###### Next, the exogenous variables are loaded in, which will be used in the HAR-X and L-HAR-X regressions

In [ ]:
data, mid_5m, logret_5m, rv_daily = build_series()

rv=rv_daily.sort_index()
idx=rv.index

#Load exogenous variables
def load_XVAR_excel(file: str, date_col: str = "observation_date") -> pd.Series:
    df = pd.read_excel(file, engine="openpyxl", decimal=",")
    if date_col not in df.columns:
        raise ValueError(f"{file}: expected a '{date_col}' column")
    value_cols = [c for c in df.columns if c != date_col]
    if not value_cols:
        raise ValueError(f"{file}: no value column besides '{date_col}'")
    value_col = value_cols[0]

    df[date_col]  = pd.to_datetime(df[date_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")

    series = (
        df.dropna(subset=[date_col, value_col])
          .sort_values(date_col)
          .drop_duplicates(subset=[date_col], keep="last")
          .set_index(date_col)[value_col]
          .rename(value_col)
    )

    return series

def align_to_idx(series):
    return series.reindex(idx).ffill()

EPU     = align_to_idx(load_XVAR_excel("EPUIndexUS.xlsx"))
FEDFUN  = align_to_idx(load_XVAR_excel("FedfundsrateUS.xlsx"))
INFL    = align_to_idx(load_XVAR_excel("InflationUS.xlsx"))
NASDAQ  = align_to_idx(load_XVAR_excel("NasdaqcompositeUS.xlsx"))
SP500   = align_to_idx(load_XVAR_excel("SP500US.xlsx"))
T10Y    = align_to_idx(load_XVAR_excel("TenyearTrateUS.xlsx"))
VIX     = align_to_idx(load_XVAR_excel("VIXUS.xlsx"))

SP500_ret  = np.log(SP500 / SP500.shift(1))
NASDAQ_ret = np.log(NASDAQ / NASDAQ.shift(1))

###### Construct one large dataset with all the variables needed in this replication

In [ ]:
# HAR components
rv_lag1  = rv.shift(1)
rv_week  = rv_lag1.rolling(5).mean()
rv_month = rv_lag1.rolling(22).mean()

# Log-HAR components
log_rv       = np.log(rv)
log_rv_lag1  = log_rv.shift(1)
log_rv_week  = log_rv_lag1.rolling(5).mean()
log_rv_month = log_rv_lag1.rolling(22).mean()

# Quarticity HARQ
n_per_day = int(logret_5m.groupby(logret_5m.index.date).size().median())
rq = ((n_per_day / 3) * logret_5m.pow(4).resample("1D").sum()).reindex(idx)
sqrt_rq_lag1 = rq.shift(1).pow(0.5)
interaction = sqrt_rq_lag1 * rv_lag1

# SHAR semivariances
pos_sq = (logret_5m.where(logret_5m > 0, 0))**2
neg_sq = (logret_5m.where(logret_5m < 0, 0))**2

rv_pos = pos_sq.resample("1D").sum().reindex(idx)
rv_neg = neg_sq.resample("1D").sum().reindex(idx)

rvpos_lag1 = rv_pos.shift(1)
rvneg_lag1 = rv_neg.shift(1)

# LevHAR
daily_ret = logret_5m.resample("1D").sum().reindex(idx)
ret_lag1   = daily_ret.shift(1)
ret_week   = ret_lag1.rolling(5).mean()
ret_month  = ret_lag1.rolling(22).mean()

retneg_lag1  = np.minimum(ret_lag1, 0)
retneg_week  = np.minimum(ret_week, 0)
retneg_month = np.minimum(ret_month, 0)

df= pd.DataFrame({

    "rv": rv,
    "logrv": log_rv,

    # HAR
    "rv_lag1": rv_lag1,
    "rv_week": rv_week,
    "rv_month": rv_month,

    # HARQ
    "sqrt_rq_lag1": sqrt_rq_lag1,
    "interaction": interaction,

    # SHAR
    "rvpos_lag1": rvpos_lag1,
    "rvneg_lag1": rvneg_lag1,

    # LevHAR
    "retneg_lag1": retneg_lag1,
    "retneg_week": retneg_week,
    "retneg_month": retneg_month,

    # Log-HAR
    "logrv_lag1": log_rv_lag1,
    "logrv_week": log_rv_week,
    "logrv_month": log_rv_month,

    # HAR‑X 
    "VIX_lag1": VIX.shift(1),
    "T10Y_lag1": T10Y.shift(1),
    "SP500_lag1": SP500_ret.shift(1),
    "NASDAQ_lag1": NASDAQ_ret.shift(1),
    "INFL_lag1": INFL.shift(1),
    "FEDFUN_lag1": FEDFUN.shift(1),
    "EPU_lag1": EPU.shift(1),
})

df = df.dropna()
df.head()

###### Check the stationarity of exogeneous regressors, as well as the multicollinearity

In [ ]:
from statsmodels.tsa.stattools import adfuller

variables = ["VIX_lag1", "T10Y_lag1", "SP500_lag1", "NASDAQ_lag1", "INFL_lag1", "FEDFUN_lag1", "EPU_lag1"]

adf_results = {}

for var in variables:
    series = df[var].dropna()
    result = adfuller(series, autolag="AIC")

    adf_results[var] = {
        "ADF Statistic": result[0],
        "p-value": result[1]   
    }

adf_df = pd.DataFrame(adf_results).T
print(adf_df)

corr_matrix = df[variables].corr()
print(corr_matrix)


###### Because SP500 and NASDAQ have high correlation, an auxillary regressions is performed, and the residuals of this regression are used instead of both variables in the HAR-X and L-HAR-X regressions

In [ ]:
nas = df["NASDAQ_lag1"]
sp  = df["SP500_lag1"]

aux_df = pd.concat([nas, sp], axis=1).dropna()
aux_df.columns = ["NASDAQ_lag1", "SP500_lag1"]

X = sm.add_constant(aux_df["SP500_lag1"])
y = aux_df["NASDAQ_lag1"]

aux_model = sm.OLS(y, X).fit()

resid_lag1 = aux_model.resid
resid_lag1.name = "resid_lag1"

df["resid_lag1"] = resid_lag1

###### As VIX, 10-year treasury rate, inflation and federal funds rate are nonstationary, the first difference of these is taken

In [ ]:
dVIX    = VIX.diff()
dT10Y   = T10Y.diff()
dINFL   = INFL.diff()
dFEDFUN = FEDFUN.diff()

df["dVIX_lag1"]     = dVIX.shift(1)
df["dT10Y_lag1"]    = dT10Y.shift(1)
df["dINFL_lag1"]    = dINFL.shift(1)
df["dFEDFUN_lag1"]  = dFEDFUN.shift(1)

diff_vars = ["dVIX_lag1", "dT10Y_lag1", "dINFL_lag1", "dFEDFUN_lag1"]

adf_results = {}

for var in diff_vars:
    series = df[var].dropna()
    result = adfuller(series, autolag="AIC")

    adf_results[var] = {
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "N Observations": result[3]
    }

adf_df = pd.DataFrame(adf_results).T

#Check if first-difference is stationary
print(adf_df)

In [ ]:
#Create test and training datasets
split_idx = int(0.8 * len(df))
train = df.iloc[:split_idx]
test  = df.iloc[split_idx:]

train_idx = train.index
test_idx  = test.index

###### Create rolling window forecast function and function to calculate mean-squared error

In [ ]:
def rolling_forecast(train_df, test_df, y_col, feature_cols):
    preds = []
    actuals = []

    y_train_series = train_df[y_col]
    X_train_block  = train_df[feature_cols]

    for i, t in enumerate(test_df.index):
        if i == 0:
            X_in = X_train_block
            y_in = y_train_series
        else:
            X_in = pd.concat([X_train_block, test_df[feature_cols].iloc[:i]])
            y_in = pd.concat([y_train_series, test_df[y_col].iloc[:i]])

        Xc = sm.add_constant(X_in, has_constant='add')
        res = sm.OLS(y_in, Xc).fit()

        X_t = sm.add_constant(test_df[feature_cols].loc[[t]], has_constant='add')
        pred = res.predict(X_t).iloc[0]

        preds.append(pred)
        actuals.append(test_df[y_col].loc[t])

    return pd.Series(preds, index=test_df.index), pd.Series(actuals, index=test_df.index)

def mse(pred, actual):
    return ((pred - actual)** 2).mean()

In [ ]:
# HAR
har_fc, har_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rv_lag1","rv_week","rv_month"]
)
print("HAR MSE:", mse(har_fc, har_ac))

# HARQ
harq_fc, harq_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rv_lag1","interaction","rv_week","rv_month"]
)
print("HARQ MSE:", mse(harq_fc, harq_ac))

# Log-HAR
log_fc_log, log_ac_log = rolling_forecast(
    train, test,
    y_col="logrv",
    feature_cols=["logrv_lag1","logrv_week","logrv_month"]
)

# Jensen correction
resid_var = (log_ac_log - log_fc_log).var()
log_fc = np.exp(log_fc_log + 0.5*resid_var)
log_ac = np.exp(log_ac_log)
print("LogHAR MSE:", mse(log_fc, log_ac))

# LevHAR
levhar_fc, levhar_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rv_lag1","rv_week","rv_month",
                  "retneg_lag1","retneg_week","retneg_month"]
)
print("LevHAR MSE:", mse(levhar_fc, levhar_ac))

# SHAR
shar_fc, shar_ac = rolling_forecast(
    train, test,
    y_col="rv",
    feature_cols=["rvneg_lag1","rvpos_lag1","rv_week","rv_month"]
)
print("SHAR MSE:", mse(shar_fc, shar_ac))

In [ ]:
print("\n HAR-X forecasts MSE")
harx_fc, harx_actual=rolling_forecast(
        train, test, y_col="rv",
        feature_cols=['rv_lag1','rv_week','rv_month',
        'dVIX_lag1', 'dT10Y_lag1', 'dINFL_lag1',
        'dFEDFUN_lag1', 'EPU_lag1', 'resid_lag1']
    )
print(mse(harx_fc, harx_actual))

# L-HAR-X forecasts and MSE
print("\n L-HAR-X forecasts MSE")
lharx_fc_log, lharx_actual_log = rolling_forecast(
    train, test,
    feature_cols=[
    "logrv_lag1", "logrv_week", "logrv_month",
    'dVIX_lag1', 'dT10Y_lag1', 'dINFL_lag1',
    'dFEDFUN_lag1', 'EPU_lag1', 'resid_lag1'],
    y_col="logrv"
    )
# Jensen correction 
resid_var=(lharx_actual_log - lharx_fc_log).var()
lharx_fc=np.exp(lharx_fc_log + 0.5 * resid_var)
lharx_actual=np.exp(lharx_actual_log)

print(mse(lharx_fc, lharx_actual))

###### Now, apply the following regularization techniques to the HAR-X regression; Ridge regression, LASSO, Elastic Net, Post LASSO, Adaptive LASSO

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet

def to_scalar(x):
    if hasattr(x, "values"):
        x = x.values
    return float(np.asarray(x).reshape(-1)[0])

def scalar_pred(model, X):
    yhat = model.predict(X)
    arr = np.asarray(yhat).ravel()
    return float(arr[0])

# Hyperparameter tuning for Ridge/Lasso/Elastic Net
def tune_regularization(model_class, X_train, y_train, X_val, y_val):
    lambdas = np.logspace(-5, 2, 80)
    alphas  = np.linspace(0, 1, 8)

    best_mse = np.inf
    best_model = None
    best_scaler = None

    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train.values)
    Xvl = scaler.transform(X_val.values)

    for lam in lambdas:
        if model_class == ElasticNet:
            for a in alphas:
                model = ElasticNet(alpha=lam, l1_ratio=a, max_iter=4000)
                model.fit(Xtr, y_train)
                mse_val = ((model.predict(Xvl) - y_val)**2).mean()
                if mse_val < best_mse:
                    best_mse = mse_val
                    best_model = model
                    best_scaler = scaler

        else:
            model = model_class(alpha=lam, max_iter=4000)
            model.fit(Xtr, y_train)
            mse_val = ((model.predict(Xvl) - y_val)**2).mean()
            if mse_val < best_mse:
                best_mse = mse_val
                best_model = model
                best_scaler = scaler

    return best_model, best_scaler

# Rolling window (train + validation) for Ridge/Lasso/Elastic Net
def rolling_regularized(model_class, X, y, train_size=0.7, val_size=0.1):
    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size) * T)
    preds, acts = [], []
    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]

        best_model, scaler = tune_regularization(model_class, X_train, y_train, X_val, y_val)

        pred = scalar_pred(best_model, scaler.transform(X_test.values))
        preds.append(pred)
        acts.append(y.iloc[t])

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size) * T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

# Adaptive Lasso
def rolling_adaptive_lasso(X, y, train_size=0.7, val_size=0.1):
    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size)*T)
    preds, acts = [], []
    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]

        Xc = sm.add_constant(X_train)
        beta = sm.OLS(y_train, Xc).fit().params[1:]   
        weights = 1 / (np.abs(beta) + 1e-6)

        Xw_train = X_train * weights.values
        Xw_val   = X_val   * weights.values
        Xw_test  = X_test  * weights.values

        best_lasso, scaler = tune_regularization(Lasso, Xw_train, y_train, Xw_val, y_val)
        pred = scalar_pred(best_lasso, scaler.transform(Xw_test.values))

        preds.append(pred)
        acts.append(y.iloc[t])

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size) * T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

# Post-Lasso
def rolling_post_lasso(X, y, train_size=0.7, val_size=0.1):

    T = len(X)
    train_end = int(train_size * T)
    val_end   = int((train_size + val_size) * T)

    preds, acts = [], []

    for t in range(val_end, T):

        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]

        X_val   = X.iloc[train_end:val_end]
        y_val   = y.iloc[train_end:val_end]

        X_test  = X.iloc[t:t+1]

        # Stage 1: tuned Lasso
        best_lasso, scaler = tune_regularization(Lasso, X_train, y_train, X_val, y_val)
        mask = best_lasso.coef_ != 0

        if mask.sum() == 0:
            pred = float(y_train.mean())

        else:
            Xs_train = X_train.iloc[:, mask].values
            Xs_test  = X_test.iloc[:, mask].values

            X_train_mat = np.column_stack([np.ones(len(Xs_train)), Xs_train])
            X_test_mat  = np.column_stack([np.ones(len(Xs_test)),  Xs_test])

            beta, *_ = np.linalg.lstsq(X_train_mat, y_train.values, rcond=None)
            pred = to_scalar(X_test_mat @ beta)  

        preds.append(pred)
        acts.append(y.iloc[t])

        train_end += 1
        val_end   += 1

    idx = X.index[int((train_size + val_size) * T):]
    return pd.Series(preds, index=idx), pd.Series(acts, index=idx)

###### Finally, we select the HAR-X features

In [ ]:
harx_features = [
    "rv_lag1","rv_week","rv_month",
    "dVIX_lag1","dT10Y_lag1","dINFL_lag1","dFEDFUN_lag1",
    "EPU_lag1","resid_lag1"
]

X = df[harx_features]
y = df["rv"]

# Regularized HAR-X Models
print("\nRIDGE HAR-X")
rr_fc, rr_act = rolling_regularized(Ridge, X, y)
print("MSE:", mse(rr_fc, rr_act))

print("\nLASSO HAR-X")
la_fc, la_act = rolling_regularized(Lasso, X, y)
print("MSE:", mse(la_fc, la_act))

print("\nELASTIC NET HAR-X")
en_fc, en_act = rolling_regularized(ElasticNet, X, y)
print("MSE:", mse(en_fc, en_act))

print("\nADAPTIVE LASSO HAR-X")
ala_fc, ala_act = rolling_adaptive_lasso(X, y)
print("MSE:", mse(ala_fc, ala_act))

print("\nPOST-LASSO HAR-X")
pl_fc, pl_act = rolling_post_lasso(X, y)
print("MSE:", mse(pl_fc, pl_act))